# Lab 14: Topología de Surface Codes

Los **surface codes** son el esquema de corrección de errores cuánticos más prometedor para hardware actual. Se basan en un retículo 2D donde:

- **Qubits de datos** almacenan la información.
- **Qubits ancilla** (medidores de síndrome) detectan errores sin medir el estado lógico.

Un código surface $d\times d$ codifica **1 qubit lógico** en $d^2$ qubits físicos y puede corregir hasta $\lfloor(d-1)/2\rfloor$ errores arbitrarios.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity, partial_trace
from qiskit.primitives import StatevectorSampler

np.random.seed(42)

## 1. Retículo del Surface Code d=3

El surface code de distancia $d=3$ requiere $9$ qubits de datos y $8$ ancillas (4 estabilizadores $X$, 4 estabilizadores $Z$). Visualizamos la topología del retículo.

In [ ]:
def draw_surface_code_d3():
    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    ax.set_xlim(-0.5, 5.5); ax.set_ylim(-0.5, 5.5)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title('Surface Code d=3 — Retículo', fontsize=14, fontweight='bold')

    # Data qubits en posiciones (1,1),(1,3),(1,5),(3,1),(3,3),(3,5),(5,1),(5,3),(5,5)
    data_pos = [(i, j) for i in [1, 3, 5] for j in [1, 3, 5]]
    for idx, (x, y) in enumerate(data_pos):
        c = plt.Circle((x, y), 0.35, color='steelblue', zorder=3)
        ax.add_patch(c)
        ax.text(x, y, f'D{idx}', ha='center', va='center', color='white',
                fontsize=9, fontweight='bold', zorder=4)

    # Z-stabilizers (plaquettes)
    z_centers = [(2, 2), (2, 4), (4, 2), (4, 4)]
    for xc, yc in z_centers:
        sq = mpatches.FancyBboxPatch((xc-0.8, yc-0.8), 1.6, 1.6,
                                      boxstyle='round,pad=0.1',
                                      facecolor='#ffd700', alpha=0.5, zorder=1)
        ax.add_patch(sq)
        ax.text(xc, yc, 'Z', ha='center', va='center', color='#8B6914',
                fontsize=11, fontweight='bold', zorder=2)

    # X-stabilizers (vertices — triangular patches en bordes)
    x_centers = [(2, 1), (4, 1), (2, 5), (4, 5)]
    for xc, yc in x_centers:
        sq = mpatches.FancyBboxPatch((xc-0.8, yc-0.45), 1.6, 0.9,
                                      boxstyle='round,pad=0.05',
                                      facecolor='#90ee90', alpha=0.6, zorder=1)
        ax.add_patch(sq)
        ax.text(xc, yc, 'X', ha='center', va='center', color='darkgreen',
                fontsize=11, fontweight='bold', zorder=2)

    # Edges between data qubits
    edges = [(0,1),(1,2),(3,4),(4,5),(6,7),(7,8),(0,3),(3,6),(1,4),(4,7),(2,5),(5,8)]
    for i, j in edges:
        x0,y0 = data_pos[i]; x1,y1 = data_pos[j]
        ax.plot([x0,x1],[y0,y1],'k-',lw=1.5,alpha=0.4,zorder=0)

    d_patch = mpatches.Patch(color='steelblue', label='Qubit de datos (9)')
    z_patch = mpatches.Patch(color='#ffd700', alpha=0.7, label='Z-estabilizador (4)')
    x_patch = mpatches.Patch(color='#90ee90', alpha=0.7, label='X-estabilizador (4)')
    ax.legend(handles=[d_patch,z_patch,x_patch], loc='lower right', fontsize=10)
    plt.tight_layout(); plt.show()

draw_surface_code_d3()
print("Surface code d=3:")
print(f"  Qubits físicos de datos: 9")
print(f"  Ancillas (estabilizadores): 8")
print(f"  Qubits lógicos codificados: 1")
print(f"  Distancia de código: 3")
print(f"  Errores corregibles: ≤1 error arbitrario")

## 2. Código de Repetición como Caso 1D

El **código de repetición** es el surface code d×1: codifica 1 qubit lógico en $d$ físicos y detecta/corrige errores de bit-flip (X). Es el punto de partida pedagógico del surface code.

In [ ]:
def encode_logical_zero(d=3):
    """Codifica |0̄⟩ = |000...0⟩ y |1̄⟩ = |111...1⟩ para el código de repetición."""
    qc = QuantumCircuit(d, name=f'|0̄⟩ d={d}')
    # |0̄⟩ ya está en |000⟩; preparar |+̄⟩ con H lógico
    for i in range(1, d):
        qc.cx(0, i)
    return qc

def syndrome_repetition(d=3):
    data = QuantumRegister(d, 'data')
    anc  = QuantumRegister(d-1, 'anc')
    cr   = ClassicalRegister(d-1, 's')
    qc = QuantumCircuit(data, anc, cr)
    for i in range(d-1):
        qc.cx(data[i], anc[i])
        qc.cx(data[i+1], anc[i])
    qc.measure(anc, cr)
    return qc

# Demostración: aplicar error en qubit 1 y detectar con síndrome
d = 3
qc_enc = encode_logical_zero(d)
qc_syn = syndrome_repetition(d)

# Error en qubit 1
qc_full = QuantumCircuit(d + d-1, d-1)
qc_full.compose(qc_enc, qubits=range(d), inplace=True)
qc_full.x(1)   # Error de bit-flip en qubit 1
qc_full.barrier()
qc_full.compose(qc_syn, inplace=True)

sampler = StatevectorSampler()
job = sampler.run([qc_full], shots=512)
counts = job.result()[0].data.s.get_counts()
print("Síndrome medido (error en qubit 1):")
for s, c in sorted(counts.items(), key=lambda x:-x[1]):
    print(f"  s={s} → {c} cuentas")

syndrome_map = {'11': 'Error en qubit 1', '10': 'Error en qubit 0',
                '01': 'Error en qubit 2', '00': 'Sin error'}
top = max(counts, key=counts.get)
print(f"\nDiagnóstico: {syndrome_map.get(top, 'desconocido')}")

## 3. Umbral de Error del Surface Code

La corrección es posible si la tasa de error física $p < p_{\text{umbral}}$. Para el surface code, $p_{\text{umbral}} \approx 1\%$. Por debajo del umbral, aumentar la distancia $d$ **reduce exponencialmente** la tasa lógica de error:

$$p_L \approx A \left(\frac{p}{p_{\text{umbral}}}\right)^{\lfloor d/2\rfloor + 1}$$

In [ ]:
# Tasa de error lógica vs distancia y tasa física
p_threshold = 0.01
A = 0.5
distances = [3, 5, 7, 9]
p_physical = np.logspace(-4, -1, 100)

fig, ax = plt.subplots(figsize=(8, 5))
for d in distances:
    t = (d - 1) // 2 + 1
    p_logical = A * (p_physical / p_threshold) ** t
    ax.semilogy(p_physical, p_logical, lw=2, label=f'd={d} (t={t-1})')

ax.axvline(p_threshold, color='red', ls='--', lw=2, label=f'Umbral p={p_threshold}')
ax.set_xlabel('Tasa de error física p', fontsize=12)
ax.set_ylabel('Tasa de error lógica $p_L$', fontsize=12)
ax.set_title('Surface Code: tasa de error lógica vs distancia')
ax.legend(fontsize=10); ax.set_xlim([1e-4, 0.1])
plt.tight_layout(); plt.show()

print("Overhead de qubits físicos por qubit lógico:")
for d in distances:
    print(f"  d={d}: {d**2} qubits físicos")